# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides an end-to-end guide for loading and exploring the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import pprint

# Define the Croissant schema URL for the FAIR2 dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json"

# Load the dataset: metadata and structure
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

# Optionally view the available metadata keys
print("Available metadata keys:", list(metadata.keys()))

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each component in Croissant (record sets, fields, columns) is referenced by its `@id`. We'll print out the record sets, and for each, the fields and columns with their `@id`.

In [ ]:
# List all record sets and their @id
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- Name: {rs.name}, @id: {rs.id}, Description: {rs.description if hasattr(rs, 'description') else 'N/A'}")

# For each record set, enumerate fields and columns by @id
for rs in record_sets:
    print(f"\nRecordSet name: {rs.name} (@id: {rs.id})")
    print("Fields:")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}, dataType: {field.data_type})")
        # Enumerate columns
        if hasattr(field, 'columns'):
            for col in field.columns:
                print(f"    Column: {col.name} (@id: {col.id}, dataType: {col.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s from the overview. We'll load all record sets found and display basic information about the main one.

In [ ]:
# Prepare to extract records by their record set @id
record_set_ids = [rs.id for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Show columns for the main record set (first one)
if dataframes:
    main_rs_id = list(dataframes.keys())[0]
    print(f"Columns for record set {main_rs_id}:")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())
else:
    print("No records loaded.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll select a numeric field based on the available columns. For example, suppose there is a column with GAD-7 score (`gad7_score`). We'll reference it by its `@id`.

*Replace `<numeric_field_id>` and `<group_field>` below with actual column names from your overview code above!*

In [ ]:
# Example EDA: filter by GAD-7 score (replace with actual column name and @id if different)
main_rs_id = list(dataframes.keys())[0] if dataframes else None
df = dataframes[main_rs_id] if main_rs_id else pd.DataFrame()

# Try to find numeric fields
numeric_fields = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
print("Numeric fields found:", numeric_fields)

# Choose a numeric field (for demonstration, pick first)
if numeric_fields:
    numeric_field = numeric_fields[0]  # Replace with actual @id if necessary
else:
    numeric_field = None

if numeric_field:
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a field, e.g., gender or village
    group_field_candidates = [c for c in df.columns if c not in numeric_fields]
    print("Possible grouping fields:", group_field_candidates)
    if group_field_candidates:
        group_field = group_field_candidates[0]
        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped data by {group_field}, mean {numeric_field}:")
            display(grouped_df.head())
else:
    print("No numeric field available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We'll plot a histogram of the numeric field and a boxplot by group, using pandas and matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field:
    # Histogram
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group
    if group_field in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi dataset (referenced via Croissant schema) was successfully loaded and explored using `mlcroissant`.
- Data was extracted and analyzed using field/column `@id`s ensuring reproducibility.
- The initial EDA revealed distributions of numeric mental health scores (e.g., GAD-7) and differences by demographic groupings.
- Further work can focus on data imputation, advanced statistical modeling, or integrating additional survey fields for richer insights.

This notebook demonstrates a standards-based workflow for FAIR and AI-ready data using Croissant and `mlcroissant`.